# Demo notebook for using maridatadownloader

# Table of content <a class="anchor" id="toc"></a>

* [1) GFS](#gfs)
    * [1.1) Create downloader object](#gfs-init)
    * [1.2) Download data as datacube](#gfs-datacube)
    * [1.3) Download data along trajectory](#gfs-trajectory)
* [2) CMEMS - Physics](#cmems)
    * [2.1) Create downloader object](#cmems-init)
    * [2.2) Download data as datacube](#cmems-datacube)
* [3) CMEMS - Waves](#cmems-waves)
    * [3.1) Create downloader object](#cmems-waves-init)
    * [3.2) Download data as datacube](#cmems-waves-datacube)

---

In [ ]:
import os
import time
from datetime import datetime, timezone, timedelta

import folium
import getpass
import numpy as np
import rioxarray
import xarray

from maridatadownloader import BoxSubset, TrajectorySubset, get_downloader

In [ ]:
%load_ext autoreload
%autoreload 2

---

# 1) GFS <a class="anchor" id="gfs"></a>
[go to TOC](#toc)

## 1.1) Create downloader object <a class="anchor" id="gfs-init"></a>
[go to TOC](#toc)

In [ ]:
gfs = get_downloader('gfs')

In [ ]:
gfs.get_xarray_dataset()

## 1.2) Download data as datacube <a class="anchor" id="gfs-datacube"></a>
[go to TOC](#toc)

In [ ]:
# Configuration for downloading as cube
time_min = "2024-09-20 11:00:00"
time_max = "2024-09-20 16:00:00"

lon_min=-180
lon_max=180
lat_min=-90
lat_max=90

height_min = 10
height_max = 10

parameters = ["Temperature_surface", "Pressure_reduced_to_MSL_msl", "Wind_speed_gust_surface", 
              "u-component_of_wind_height_above_ground", "v-component_of_wind_height_above_ground"]

subset = BoxSubset(time=slice(time_min, time_max))
#subset = BoxSubset(time=slice(time_min, time_max), extra={'height_above_ground': slice(height_min, height_max)})

In [ ]:
dataset_gfs = gfs.get_xarray_dataset(parameters=parameters, subset=subset)
dataset_gfs

In [ ]:
if 'sigma' in dataset_gfs.coords:
    dataset_gfs = dataset_gfs.isel(sigma=0).drop('sigma')
if 'height_above_ground' in dataset_gfs.coords and 'height_above_ground' not in subset.extra:
    dataset_gfs = dataset_gfs.sel(height_above_ground=10).drop('height_above_ground')
if 'reftime' in dataset_gfs.coords:
    dataset_gfs = dataset_gfs.drop('reftime')
if 'LatLon_Projection' in dataset_gfs.coords:   
    dataset_gfs = dataset_gfs.drop('LatLon_Projection')
dataset_gfs

In [ ]:
u_wind = gfs.get_xarray_dataset('u-component_of_wind_height_above_ground')['u-component_of_wind_height_above_ground']
u_wind.sel(height_above_ground=10).isel(time=0).plot()

### Calculate difference 

In [ ]:
diff = u_wind.sel(height_above_ground=10).isel(time=0) - u_wind.sel(height_above_ground=80).isel(time=0)

In [ ]:
diff.plot()

## 1.3) Download data along trajectory <a class="anchor" id="gfs-trajectory"></a>
[go to TOC](#toc)

References:
 - https://docs.xarray.dev/en/latest/user-guide/interpolation.html#advanced-interpolation
 - https://earth-env-data-science.github.io/lectures/xarray/xarray-part2.html
 - https://xesmf.readthedocs.io/en/stable/
 - https://stackoverflow.com/questions/56144678/xarray-point-interpolation-on-multiple-dimensions
 - https://stackoverflow.com/questions/72179103/xarray-select-the-data-at-specific-x-and-y-coordinates

In [ ]:
# Configuration for downloading along trajectory
lons = [2.81, 3.19, 4.56, 6.67, 6.68]
lats = [51.9, 53.0, 54.0, 54.3, 55.5]

times = [datetime.strptime("2023-09-20 09:00:00", '%Y-%m-%d %H:%M:%S'),
         datetime.strptime("2023-09-20 11:00:00", '%Y-%m-%d %H:%M:%S'),
         datetime.strptime("2023-09-20 13:00:00", '%Y-%m-%d %H:%M:%S'),
         datetime.strptime("2023-09-20 15:00:00", '%Y-%m-%d %H:%M:%S'),
         datetime.strptime("2023-09-20 17:00:00", '%Y-%m-%d %H:%M:%S')]

#times = ["2023-09-28 09:00:00",
#         "2023-09-28 11:00:00",
#         "2023-09-28 13:00:00",
#         "2023-09-28 15:00:00",
#         "2023-09-28 17:00:00"]

height = 10

parameters = ["Temperature_surface", "Pressure_reduced_to_MSL_msl", "Wind_speed_gust_surface", 
              "u-component_of_wind_height_above_ground", "v-component_of_wind_height_above_ground"]

subset = TrajectorySubset(time=times, latitude=lats, longitude=lons, method='linear',
                          extra={'height_above_ground': height})

In [ ]:
%%time
dataset_gfs_trajectory = gfs.get_xarray_dataset(parameters=parameters, subset=subset)
dataset_gfs_trajectory

In [ ]:
dataset_gfs_trajectory.Temperature_surface.plot();

In [ ]:
df_trajectory = dataset_gfs_trajectory.to_dataframe()
df_trajectory

In [ ]:
m = folium.Map(location=(np.mean(lats), np.mean(lons)), zoom_start=6)

for idx, row in df_trajectory.iterrows():
    html = f"""
    <b>Point {idx+1}<b/>
    <table style="width:100%">
      <tr>
        <th>Latitude</th>
        <th>{row['latitude']}</th>
      </tr>
      <tr>
        <th>Longitude</th>
        <th>{row['longitude']}</th>
      </tr>
      <tr>
        <th>Time</th>
        <th>{row['time']}</th>
      </tr>
      <tr>
        <th>Temperature_surface</th>
        <th>{row['Temperature_surface']}</th>
      </tr>
      <tr>
        <th>Pressure_reduced_to_MSL_msl</th>
        <th>{row['Pressure_reduced_to_MSL_msl']}</th>
      </tr>
      <tr>
        <th>Wind_speed_gust_surface</th>
        <th>{row['Wind_speed_gust_surface']}</th>
      </tr>
      <tr>
        <th>u-component_of_wind_height_above_ground</th>
        <th>{row['u-component_of_wind_height_above_ground']}</th>
      </tr>
      <tr>
        <th>v-component_of_wind_height_above_ground</th>
        <th>{row['v-component_of_wind_height_above_ground']}</th>
      </tr>
    </table>
    """
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        tooltip=html,
        popup=html,
    ).add_to(m)
m

In [ ]:
#idx = 0
#point = gfs.dataset.Temperature_surface.sel(
#    time=dataset_gfs_trajectory['trajectory'].time[idx].values,
#    lon=dataset_gfs_trajectory['trajectory'].lon[idx].values,
#    lat=dataset_gfs_trajectory['trajectory'].lat[idx].values,
#    method='nearest'
#)
#print(point.values)
#print(dataset_gfs_trajectory.Temperature_surface.values[idx])
#point

In [ ]:
dataset_gfs_trajectory.Temperature_surface.plot(x='time');

---

# 2) CMEMS - Physics <a class="anchor" id="cmems"></a>
[go to TOC](#toc)

## 2.1) Create downloader object <a class="anchor" id="cmems-init"></a>
[go to TOC](#toc)

In [ ]:
USERNAME = '<insert-username>'
PASSWORD = getpass.getpass('Enter your password: ')

In [ ]:
#product = 'cmems_mod_glo_phy_anfc_merged-uv_PT1H-i'  # currents
#product = 'cmems_mod_glo_phy_anfc_0.083deg_PT1H-m'  # phyiscs
product = 'cmems_mod_glo_phy_my_0.083_P1D-m' # phyiscs, my
#product = 'cmems_mod_glo_wav_anfc_0.083deg_PT3H-i'  # waves
#product = 'cmems_obs-wind_glo_phy_nrt_l4_0.125deg_PT1H' # wind

In [ ]:
cmems = get_downloader('cmems', dataset_id=product, username=USERNAME, password=PASSWORD)

## 2.2) Download data as datacube <a class="anchor" id="cmems-datacube"></a>
[go to TOC](#toc)

In [ ]:
time_min = "2023-08-29T02:30:00"
time_max = "2023-08-29T02:30:00"

lon_min=-180
lon_max=180
lat_min=-90
lat_max=90

#parameters = ["VHM0", "VMDR", "VTPK"]
#parameters = ["uo","vo","vsdx","vsdy","utide","vtide","utotal","vtotal"]
parameters = ["so", "zos", "thetao"]

subset = BoxSubset(time=slice(time_min, time_max))

In [ ]:
#cmems.dataset = cmems.dataset.sortby('time')

In [ ]:
dataset_cmemcs = cmems.get_xarray_dataset(parameters=parameters, subset=subset)
dataset_cmemcs

In [ ]:
dataset_cmemcs

### Check ocean currents contributions

In [ ]:
diff = dataset_cmemcs['uo'].isel(time=0) + dataset_cmemcs['vsdx'].isel(time=0) + dataset_cmemcs['utide'].isel(time=0) - dataset_cmemcs['utotal'].isel(time=0)

In [ ]:
diff.plot()

# 3) CMEMS - Waves <a class="anchor" id="cmems-waves"></a>
[go to TOC](#toc)

## 3.1) Create downloader object <a class="anchor" id="cmems-waves-init"></a>
[go to TOC](#toc)

In [ ]:
USERNAME = '<insert-username>'
PASSWORD = getpass.getpass('Enter your password: ')

In [ ]:
# Init downloader object
product = "cmems_mod_glo_wav_anfc_0.083deg_PT3H-i"
cmems_waves = get_downloader('cmems', dataset_id=product, username=USERNAME, password=PASSWORD)


## 3.2) Download data as datacube <a class="anchor" id="cmems-waves-datacube"></a>
[go to TOC](#toc)

In [ ]:
# Set time and coords values
time_min = "2023-01-01T02:30:00"
time_max = "2023-01-02T02:30:00"

lon_min=-180
lon_max=180
lat_min=-80
lat_max=90


# Set subset and parameters (variable names of the dataset, e.g. VMDR = sea_surface_wave_from_direction)
subset = BoxSubset(
    longitude=slice(lon_min, lon_max),
    latitude=slice(lat_min, lat_max),
    time=slice(time_min, time_max),
)

parameters=['VMDR']

In [ ]:
dataset_cmems_waves = cmems_waves.get_xarray_dataset(parameters=parameters, subset=subset)
dataset_cmems_waves